# Forecast Like-Game Clustering

This notebook runs Stage 1 of the workflow from the clean joined dataset: build game profiles, create weighted clustering features, fit the hierarchical clustering tree, render a dendrogram, and produce the new-game to like-game mapping table.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipelines.forecast_clustering import (
    build_game_profiles,
    build_weighted_feature_matrix,
    find_like_games,
    fit_hierarchical_clustering,
    load_forecast_clustering_config,
    plot_dendrogram,
    select_clustering_roles,
)

In [8]:
p=r"C:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\game_attr_cleaned.csv"
df=pd.read_csv(p)
df

,theme_sk,theme_nk,source_system_cd,source_company_cd,theme_name,theme_name_friendly,family_name,math_model_family,product_family,game_type,...,audit_created_by,audit_created_date,audit_modified_by,audit_modified_date,audit_is_deleted,game_matrix_bingo,game_matrix_hhr_exacta,game_matrix_lottery,game_matrix_prize_first_slot,game_matrix_reels_first_slot
0,1,0000,ATLASSIAN,EVERI,InitialRelease_Template_GameTitle,InitialRelease_Template_GameTitle,NaN,NaN,Standard Video,Not Banked,...,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,False,False,False,False,False,False
1,2,10000,ATLASSIAN,EVERI,Tourney Now Star Struck,Tourney Now Star Struck,NaN,['Reels First Slot'],TE ORT,Not Banked,...,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,False,False,False,False,False,True
2,3,10003,ATLASSIAN,EVERI,Little Shop of Horrors - Flex HHR,Little Shop of Horrors - Flex HHR,NaN,['HHR [Exacta]'],Standard Video,Not Banked,...,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,False,False,True,False,False,False
3,4,10005,ATLASSIAN,EVERI,Great Tiger Gold,Great Tiger Gold,NaN,"['Bingo', 'HHR [Exacta]', 'Lottery', 'Prize Fi...",Standard Video,Not Banked,...,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,False,True,True,True,True,False
4,5,10006,ATLASSIAN,EVERI,Octopus Gold,Octopus Gold,NaN,"['Bingo', 'HHR [Exacta]', 'Lottery', 'Prize Fi...",Standard Video,Not Banked,...,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,False,True,True,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4389,4390,8175,ATLASSIAN,EVERI,Tourney Halloween - Pop N Win,Tourney Halloween - Pop N Win,NaN,['Lottery'],TE ORT,Not Banked,...,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,False,False,False,True,False,False
4390,4391,8176,ATLASSIAN,EVERI,Tourney Thanksgiving - Pop N Win,Tourney Thanksgiving - Pop N Win,NaN,['Lottery'],TE ORT,Not Banked,...,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,False,False,False,True,False,False
4391,4392,8177,ATLASSIAN,EVERI,Tourney Christmas - Pop N Win,Tourney Christmas - Pop N Win,NaN,['Lottery'],TE ORT,Not Banked,...,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,False,False,False,True,False,False
4392,4393,8178,ATLASSIAN,EVERI,Tourney Bunch O' Luck - Pop N Win,Tourney Bunch O' Luck - Pop N Win,NaN,['Lottery'],TE ORT,Not Banked,...,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,1e787607-5ff7-4bfc-9ebc-694c2e5869f9,2026-06-23T17:07:49.405Z,False,False,False,True,False,False


## 1. Load Config And Clean Joined Data

In [7]:
CONFIG_PATH = ROOT / 'config' / 'forecast_clustering' / 'default.yml'
config, _ = load_forecast_clustering_config(CONFIG_PATH)

joined = pd.read_csv(config['input_path'], low_memory=False)
joined.shape
joined

,yearmonth,game_name,supplier,slot_cabinet_name,country,region,operator_type,own_status,game_category,game_classification,...,sales_first_copa_product_type,sales_first_copa_product_technology,sales_first_copa_product_style,sales_first_copa_product_series,sales_beginning_month_date_count,sales_sales_order_number_count,sales_customer_number_count,sales_reference_number_count,sales_first_beginning_month_date,sales_last_beginning_month_date
0,2022-01-01,1421 voyages of zheng he,igt,crystaldual,NaN,NaN,canada casinos,owned,video reel,class 3,...,exclude,exclude,exclude,exclude,2,2,2,2,2017-05-01,2017-12-01
1,2022-01-01,1421 voyages of zheng he,igt,crystaldual,NaN,NaN,u.s. tribal casino,owned,video reel,class 3,...,exclude,exclude,exclude,exclude,2,2,2,2,2017-05-01,2017-12-01
2,2022-01-01,1421 voyages of zheng he,igt,crystalslant,NaN,NaN,u.s. tribal casino,owned,video reel,class 3,...,exclude,exclude,exclude,exclude,2,2,2,2,2017-05-01,2017-12-01
3,2022-01-01,1421 voyages of zheng he,igt,prodigi vu,NaN,NaN,canada casinos,owned,video reel,class 3,...,exclude,exclude,exclude,exclude,2,2,2,2,2017-05-01,2017-12-01
4,2022-01-01,1421 voyages of zheng he,igt,prodigi vu,NaN,NaN,cruise ship casino,owned,video reel,class 3,...,exclude,exclude,exclude,exclude,2,2,2,2,2017-05-01,2017-12-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46015,2026-04-01,samurai 888 spin - sakura,igt,peak slant 49,united states of america,u.s. & canada,cruise ship casino,owned,video reel,class 3,...,exclude,exclude,exclude,exclude,2,2,2,2,2025-02-01,2025-03-01
46016,2026-04-01,samurai 888 spin - sakura,igt,peak slant 49,united states of america,u.s. & canada,u.s. commercial casino,owned,video reel,class 3,...,exclude,exclude,exclude,exclude,2,2,2,2,2025-02-01,2025-03-01
46017,2026-04-01,samurai 888 spin - sakura,igt,peak slant 49,united states of america,u.s. & canada,u.s. tribal casino,leased,video reel,class 2,...,exclude,exclude,exclude,exclude,2,2,2,2,2025-02-01,2025-03-01
46018,2026-04-01,samurai 888 spin - sakura,igt,peak slant 49,united states of america,u.s. & canada,u.s. tribal casino,leased,video reel,class 3,...,exclude,exclude,exclude,exclude,2,2,2,2,2025-02-01,2025-03-01


## 2. Build One Game/Cabinet Profile Per Clustered Item

In [3]:
profiles = build_game_profiles(joined, config)
profiles[['profile_id', 'theme_key', 'game_name', 'mapping_cabinet_desc', 'release_date', 'history_months']].head(10)

,profile_id,theme_key,game_name,mapping_cabinet_desc,release_date,history_months
0,0,4gt,majestic stallion,peak slant 49,2019-08-01,29
1,1,a28,double mystical mermaid,s2000,2018-01-01,50
2,2,bcf,big city 5's,s3000,2018-03-01,46
3,3,bgf,cleopatra 2,crystaldual,2017-09-01,46
4,4,bgf,cleopatra 2,crystaldual 27,2017-09-01,32
5,5,bgf,cleopatra 2,crystalslant,2017-09-01,31
6,6,bgf,cleopatra 2,peak slant 32,2017-09-01,49
7,7,bsv,honey bucks,crystalcore,2020-09-01,3
8,8,bsv,honey bucks,crystalcurve,2020-09-01,52
9,9,bsv,honey bucks,crystaldual 27,2020-09-01,7


## 3. Create Weighted Feature Matrix

In [4]:
features = build_weighted_feature_matrix(profiles, config)
features.shape, features.iloc[:, :10].head()

((450, 121),
    supplier_aristocrat  supplier_igt  mapping_cabinet_desc_avp slant  \
 0                  0.0           1.0                             0.0   
 1                  0.0           1.0                             0.0   
 2                  0.0           1.0                             0.0   
 3                  0.0           1.0                             0.0   
 4                  0.0           1.0                             0.0   
 
    mapping_cabinet_desc_avp upright  mapping_cabinet_desc_axxis upright  \
 0                               0.0                                 0.0   
 1                               0.0                                 0.0   
 2                               0.0                                 0.0   
 3                               0.0                                 0.0   
 4                               0.0                                 0.0   
 
    mapping_cabinet_desc_cobalt  mapping_cabinet_desc_crystalcore  \
 0                  

## 4. Select Target Games And Qualified Historical Candidates

In [ ]:
roles = select_clustering_roles(profiles, config)
role_summary = roles['role'].value_counts().rename_axis('role').reset_index(name='profiles')
target_preview = profiles.join(roles[['role']]).query("role == 'target'")[[
    'profile_id', 'theme_key', 'game_name', 'mapping_cabinet_desc', 'release_date', 'history_months'
]]
display(role_summary)
display(target_preview)

## 5. Fit Complete-Linkage Hierarchical Clustering

In [ ]:
linkage_matrix = fit_hierarchical_clustering(features, config)
linkage_matrix[:5]

## 6. Dendrogram Presentation

In [ ]:
dendrogram_path = plot_dendrogram(
    profiles.join(roles[['role']]),
    linkage_matrix,
    config['dendrogram_output_path'],
    config,
)
display(Image(filename=dendrogram_path))
dendrogram_path

## 7. Like-Game Mapping Table

In [ ]:
like_games = find_like_games(
    profiles=profiles,
    feature_matrix=features,
    linkage_matrix=linkage_matrix,
    roles=roles,
    config=config,
)

like_games.sort_values(['target_game_name', 'target_mapping_cabinet_desc', 'candidate_rank']).head(30)

## 8. Save Notebook Outputs

In [ ]:
profiles.join(roles[['is_target', 'is_qualified_historical', 'role']]).to_csv(
    config['profiles_output_path'], index=False
)
features.to_csv(config['feature_matrix_output_path'], index=False)
like_games.to_csv(config['like_games_output_path'], index=False)

{
    'profiles': config['profiles_output_path'],
    'features': config['feature_matrix_output_path'],
    'like_games': config['like_games_output_path'],
    'dendrogram': dendrogram_path,
}